# 第 7 章：进阶特性——dataclass、组合、Mixin、`__new__`、描述符与元类

> 本章目标：掌握现代 Python OOP 的高频武器（`dataclass`、组合、Mixin），了解 `__new__`、描述符、元类这些"框架级"机制的原理，最后用 **SOLID 原则**串联全局。

---
## 7.1 `@dataclass`：告别样板代码

定义"数据为主"的类（坐标、配置、记录……）时，手写 `__init__`/`__repr__`/`__eq__` 重复而无聊。`dataclasses` 装饰器自动生成它们：

In [1]:
from dataclasses import dataclass, field


@dataclass
class Point:
    x: float
    y: float = 0.0        # 有默认值，自动进 __init__


# 一行没写，自动获得：__init__、__repr__、__eq__
p1 = Point(3, 4)
p2 = Point(3, 4)
print(p1)              # Point(x=3, y=4)
print(p1 == p2)        # True（按字段值比较）


@dataclass(order=True)  # order=True 再送你 < <= > >= 全套比较
class Task:
    priority: int
    name: str = field(compare=False)   # name 不参与比较


tasks = [Task(3, "写文档"), Task(1, "修 Bug"), Task(2, "开会")]
for t in sorted(tasks):
    print(t)

Point(x=3, y=4)
True
Task(priority=1, name='修 Bug')
Task(priority=2, name='开会')
Task(priority=3, name='写文档')


### 可变默认值：`field(default_factory=...)`

还记得第 1 章"可变类属性"陷阱吗？dataclass 里同样存在：`tags: list = []` 直接报错（ValueError），必须用工厂：

In [2]:
from dataclasses import dataclass, field


@dataclass
class Student:
    name: str
    scores: list = field(default_factory=list)   # 每个实例一个新列表

    @property
    def average(self):
        return sum(self.scores) / len(self.scores) if self.scores else 0


s1, s2 = Student("张三"), Student("李四")
s1.scores.extend([90, 80])
print(s1.scores, s2.scores)      # 互不影响
print(s1.name, s1.average)

# 试试直接写 scores: list = [] 会发生什么
try:
    exec("""
@dataclass
class Bad:
    items: list = []
""")
except ValueError as e:
    print(f"dataclass 帮你拦下了陷阱: {e}")

[90, 80] []
张三 85.0
dataclass 帮你拦下了陷阱: mutable default <class 'list'> for field items is not allowed: use default_factory


### dataclass 常用参数速查

| 参数 | 作用 |
|------|------|
| `init=True` | 生成 `__init__`（设 False 可以手写） |
| `repr=True` | 生成 `__repr__` |
| `eq=True` | 生成 `__eq__`（按字段比较） |
| `order=False` | 生成 `< <= > >=` |
| `frozen=True` | **不可变**（赋值报错，可哈希，可当 dict key） |
| `slots=True` | 生成 `__slots__`（3.10+，省内存） |

> 💡 也提一句 `@dataclass` 的兄弟：**`typing.NamedTuple`**（不可变、是 tuple 子类）与 **`attrs`**（第三方，功能更多）。

In [3]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int


red = Color(255, 0, 0)
print({red: "红色"})       # frozen -> 可哈希 -> 能当 dict key

try:
    red.r = 0              # frozen -> 禁止修改
except Exception as e:
    print(f"不可变拦截: {type(e).__name__}: {e}")

{Color(r=255, g=0, b=0): '红色'}
不可变拦截: FrozenInstanceError: cannot assign to field 'r'


## 7.2 组合优于继承
继承解决 "**is-a**"，组合解决 "**has-a**"。继承被滥用的项目，往往应该用组合重构：

```mermaid
flowchart TD
    subgraph 继承[❌ 滥用继承]
        Car --> Engine
        Car --> Wheel
        Car --> Seat
    end
    subgraph 组合[✅ 组合]
        C[Car] -- has-a --> E[Engine]
        C -- has-a --> W[Wheel x4]
        C -- has-a --> S[Seat x5]
    end
```

| 对比 | 继承 | 组合 |
|------|------|------|
| 关系 | is-a（是一个） | has-a（有一个） |
| 耦合 | **白盒**：子类依赖父类实现细节，父类一改全炸 | **黑盒**：只依赖接口 |
| 灵活性 | 编译期（定义时）固定 | 运行期可替换部件 |
| 深度 | 继承链超过 2~3 层就该警惕 | 层级扁平 |

> 经验法则：先想组合；继承只在清晰的 is-a 且确有行为复用时使用。

In [4]:
# 组合示例：机器人由多个"能力"组成，可插拔
class LaserArm:
    def attack(self):
        return "激光射击！"

class MissilePod:
    def attack(self):
        return "导弹齐射！"

class Robot:
    def __init__(self, name, weapon):
        self.name = name
        self.weapon = weapon          # 组合：武器是个独立对象，随时可换

    def fight(self):
        return f"{self.name}: {self.weapon.attack()}"


r1 = Robot("先锋", LaserArm())
print(r1.fight())

r1.weapon = MissilePod()             # 运行时换"零件"，继承做不到
print(r1.fight())

先锋: 激光射击！
先锋: 导弹齐射！


## 7.3 Mixin：正道的光（多继承的正确打开方式）

**Mixin** = 只提供一组**方法**（通常不定义 `__init__`、不存状态）的小类，通过多继承"混入"功能。

```mermaid
classDiagram
    class JSONMixin {
        +to_json() str
    }
    class LoggingMixin {
        +log(msg)
    }
    class BaseModel {
        +data
    }
    class User {
    }
    BaseModel <|-- User
    JSONMixin <|.. User : 混入
    LoggingMixin <|.. User : 混入
```

命名惯例：以 `Mixin` 结尾；排在继承列表**主基类之前**（`class User(JSONMixin, LoggingMixin, BaseModel)`）。

In [5]:
import json
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")


class JSONMixin:
    """提供 JSON 序列化能力（要求宿主有 to_dict 方法）"""
    def to_json(self):
        return json.dumps(self.to_dict(), ensure_ascii=False)


class LoggingMixin:
    """提供日志能力"""
    def log(self, msg):
        logging.info(f"[{type(self).__name__}] {msg}")


class BaseModel:
    def __init__(self, **data):
        self.data = data

    def to_dict(self):
        return self.data


class User(JSONMixin, LoggingMixin, BaseModel):   # Mixin 在前，主基类在后
    pass


u = User(name="张三", age=18)
print(u.to_json())     # 来自 JSONMixin
u.log("用户创建成功")   # 来自 LoggingMixin
print(u.data)          # 来自 BaseModel

# Mixin 可自由组合，搭出不同的类
class AuditLog(JSONMixin, BaseModel):
    pass

print(AuditLog(event="login").to_json())

[User] 用户创建成功


{"name": "张三", "age": 18}
{'name': '张三', 'age': 18}
{"event": "login"}


## 7.4 `__new__` vs `__init__`：对象是怎么诞生的

- `__new__(cls, ...)`：**创建并返回**实例（先执行）；是静态方法，返回值就是新对象
- `__init__(self, ...)`：**初始化**已创建的实例（后执行）；必须返回 None

```mermaid
sequenceDiagram
    participant C as 调用 ClassName(...)
    participant N as __new__
    participant I as __init__
    C->>N: 分配内存 创建空对象
    N-->>C: 返回实例 obj
    C->>I: 用 obj 作为 self 初始化
    I-->>C: 完成（隐式返回 None）
```

`__new__` 的经典应用：**单例模式**、**缓存/复用实例**（如 `bool`、小整数为什么 `is` 判断能成立）：

In [6]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)   # 真正创建对象
        return cls._instance                        # 已存在则直接复用

    def __init__(self, value):
        self.value = value


a = Singleton("第一次")
b = Singleton("第二次")
print(a is b)        # True -- 始终同一个对象
print(b.value)       # "第二次"（__init__ 每次都会执行！）

# 实例缓存的例子：同一个 key 复用对象
class Interned:
    _pool = {}

    def __new__(cls, key):
        if key not in cls._pool:
            cls._pool[key] = super().__new__(cls)
        return cls._pool[key]


print(Interned("a") is Interned("a"))   # True

True
第二次
True


## 7.5 描述符：`@property` 背后的原理
事实上，**`property`、`classmethod`、`staticmethod` 本身都是描述符**。理解它就看懂了 Python 属性访问的"底层魔法"：

**描述符协议**：一个类定义了 `__get__` / `__set__` / `__delete__` 中的任意一个，它的实例放在**另一个类的类属性**上时，就获得了拦截属性读写的能力。

| 类型 | 实现的方法 | 谁能覆盖同名实例属性 |
|------|-----------|-------------------|
| **数据描述符** | `__get__` + `__set__` | 类属性优先级**高于**实例 `__dict__` |
| **非数据描述符** | 只有 `__get__` | 实例 `__dict__` 优先（如方法、property 之外的普通函数） |

```mermaid
flowchart TD
    A[obj.attr] --> B{attr 是数据描述符?}
    B -- 是 --> C[调用描述符的 __get__]
    B -- 否 --> D{实例 __dict__ 有?}
    D -- 有 --> E[返回实例属性]
    D -- 没有 --> F{是类属性?}
    F -- 描述符 --> C
    F -- 普通值 --> G[返回类属性]
    F -- 没有 --> H[AttributeError]
```

自己实现一个"带校验的字段"描述符--这正是 **ORM（如 Django Model）字段**的工作原理：

In [7]:
class ValidatedField:
    """数据描述符：带类型校验的字段"""

    def __init__(self, field_type, name):
        self.field_type = field_type
        self.name = name          # 存储用的内部属性名

    def __get__(self, instance, owner):
        if instance is None:                    # 通过类访问时（如 User.age）
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if not isinstance(value, self.field_type):
            raise TypeError(f"{self.name} 需要 {self.field_type.__name__}，"
                            f"得到 {type(value).__name__}")
        instance.__dict__[self.name] = value


class User:
    age = ValidatedField(int, "age")       # 类属性上放描述符实例
    name = ValidatedField(str, "name")


u = User()
u.age = 18            # 走 __set__，校验通过
u.name = "张三"
print(u.name, u.age)  # 走 __get__

try:
    u.age = "十八岁"   # 走 __set__，校验失败
except TypeError as e:
    print(f"拦截: {e}")

张三 18
拦截: age 需要 int，得到 str


## 7.6 元类：类的类
万物皆对象--**类本身也是对象**，它也有自己的类型，就是**元类（metaclass）**：

```mermaid
flowchart LR
    I[实例 obj] -- type --> C[类 Dog] -- type --> M[元类 type] -- type --> M
    C -- 创建 --> I
    M -- 创建 --> C
```

- 普通类的元类默认是 `type`；
- `class Dog:` 语句本质上是执行 `type("Dog", (object,), {...})`；
- 自定义元类能**在类创建时**审查/修改/注册类--ORM 基类（Django `Model`）、ABC 的 `ABCMeta`、协议插件注册都是这么实现的。

> ⚠️ Tim Peters 名言："如果你不确定是否需要元类，那你就**不需要**。"（能用装饰器/`__init_subclass__` 解决就别上元类。）

In [8]:
# 1) type 的三参数形式：动态造类
Dog = type("Dog", (object,), {"bark": lambda self: "汪汪", "legs": 4})
d = Dog()
print(d.bark(), d.legs)

# 2) 轻量替代：__init_subclass__ 钩子（多数场景够用，不需要元类）
class Plugin:
    registry = {}

    def __init_subclass__(cls, **kwargs):
        """每个子类定义时自动执行"""
        super().__init_subclass__(**kwargs)
        Plugin.registry[cls.__name__] = cls


class CSVPlugin(Plugin): pass
class JSONPlugin(Plugin): pass

print(Plugin.registry)   # 定义即注册！

# 3) 真正的元类（眼熟即可）
class AutoRegister(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.created_at = "由元类注入的属性"
        return cls


class MyClass(metaclass=AutoRegister): pass
print(MyClass.created_at)

汪汪 4
{'CSVPlugin': <class '__main__.CSVPlugin'>, 'JSONPlugin': <class '__main__.JSONPlugin'>}
由元类注入的属性


## 7.7 SOLID 原则在 Python 中的落点
OOP 语法只是工具，设计原则才是灵魂。经典 SOLID 五原则与本章知识的对应：

| 原则 | 含义 | Python 落点 |
|------|------|------------|
| **S** 单一职责 | 一个类只做一件事 | 拆分 Mixin（7.3）、组合（7.2） |
| **O** 开闭原则 | 对扩展开放，对修改关闭 | 多态 + 鸭子类型（第 4 章）：加新类型不改老代码 |
| **L** 里氏替换 | 子类必须能无缝替换父类 | 继承重写不改变契约（第 3 章）；ABC 定义契约（第 6 章） |
| **I** 接口隔离 | 接口小而专，不强迫实现用不到的方法 | `Protocol`（第 4 章）拆多个小协议 |
| **D** 依赖倒置 | 依赖抽象而非具体实现 | 面向 ABC / Protocol 编程（第 4、6 章） |

```mermaid
mindmap
  root((Python OOP 学习地图))
    基础
      类与对象 self __init__
    封装
      约定 _x __x
      property slots
    复用
      继承 MRO
      组合 Mixin
    多态
      鸭子类型
      ABC Protocol
    协议
      魔术方法
    现代工具
      dataclass
    深水区
      描述符 元类
      __new__
```

## 7.8 全系列总结

| 章 | 主题 | 一句话 |
|----|------|--------|
| 01 | 类与对象 | 图纸与房子；self 是实例自身 |
| 02 | 封装 | 命名约定 + property；没有真私有 |
| 03 | 继承 | is-a；super 按 MRO 找下一个 |
| 04 | 多态 | 关心能做什么而非是什么 |
| 05 | 魔术方法 | 实现协议，融入语言本身 |
| 06 | 三种方法与 ABC | self/cls/无参三选一；契约用 ABC |
| 07 | 进阶 | dataclass 提效；组合优先；元类慎用 |

### 📝 综合练习（毕业设计）

实现一个**简易图书管理系统**，要求用上全系列知识：
1. `@dataclass` 定义 `Book`（不可变）；
2. `Library` 实现 `__len__`/`__getitem__`/`__iter__`（容器协议）、`__enter__`/`__exit__`（借阅事务回滚）、`__call__`（快捷查询）；
3. 用 ABC 定义 `Notifier`，实现 `EmailNotifier`/`SmsNotifier` 多态通知；
4. 用 Mixin 混入 `JsonExportMixin`、`LoggingMixin`；
5. 用描述符给 `Member.borrow_limit` 做类型与范围校验；
6. （加分）用 `__init_subclass__` 自动注册新的通知渠道。

祝你在 Python OOP 的路上越走越远！🐍